# 1. Initialization

In [ ]:
import sys
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import reduce

from common.helpers import get_bronze

spark = SparkSession.builder.getOrCreate()

batch_id = dbutils.widgets.get("batch_id")

print(f"[INFO] Batch ID: {batch_id}")

# 2. Read Bronze license_keys

In [ ]:
bronze_path = "/Volumes/datalake_catalog/datalake_schema/bronze/license_keys"

df_bronze = get_bronze(bronze_path, spark=spark)
df_bronze.show(5)

# 3. Deduplication (Business Rule)

In [ ]:
w = Window.partitionBy("license_id", "issued_date").orderBy(F.col("ingest_time").desc())

df_ranked = df_bronze.withColumn("rn", F.row_number().over(w))

df_clean = df_ranked.filter(F.col("rn") == 1).drop("rn")

df_quarantine_dup = df_ranked.filter(F.col("rn") > 1).drop("rn")

# 4. Type Casting

In [ ]:
df_cast = df_clean.select(
    F.col("license_id").cast("bigint"),
    F.col("subscription_id").cast("bigint"),
    F.col("max_seats").cast("int"),
    F.col("issued_date").cast("date"),
    F.col("expiry_date").cast("date"),
    F.col("ingest_time").cast("timestamp")
)

# 5. NULL Validation

In [ ]:
invalid = (
    F.col("license_id").isNull() |
    F.col("subscription_id").isNull() |
    F.col("issued_date").isNull() |
    F.col("expiry_date").isNull()
)

df_valid_null = df_cast.filter(~invalid)
df_quarantine_null = df_cast.filter(invalid)

# 6. Read Silver Subscriptions (Referential Integrity)

In [ ]:
silver_sub_path = "/Volumes/datalake_catalog/datalake_schema/silver/subscriptions"

df_subs = spark.read.format("delta").load(silver_sub_path)

In [ ]:
df_valid_fk = df_valid_null.join(
    df_subs.select("subscription_id").distinct(),
    "subscription_id",
    "inner"
)

df_quarantine_fk = df_valid_null.join(
    df_subs.select("subscription_id").distinct(),
    "subscription_id",
    "left"
).filter(F.col("subscription_id").isNull())

# 7. Domain Rules

In [ ]:
df_domain_valid = df_valid_fk.filter(F.col("max_seats").between(5, 500))
df_quarantine_domain = df_valid_fk.filter(~F.col("max_seats").between(5, 500))

# 8. Date Logic Validation

In [ ]:
df_final = df_domain_valid.filter(F.col("expiry_date") > F.col("issued_date"))

df_quarantine_date = df_domain_valid.filter(F.col("expiry_date") <= F.col("issued_date"))

# 9. Combine All Quarantines

In [ ]:
df_quarantine_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    [
        df_quarantine_dup,
        df_quarantine_null,
        df_quarantine_fk,
        df_quarantine_domain,
        df_quarantine_date
    ]
)

# 10. Write to Silver Delta (Upsert)

In [ ]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/license_keys"

w = Window.partitionBy("license_id").orderBy(F.col("ingest_time").desc())

df_upsert = (
    df_final
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if DeltaTable.isDeltaTable(spark, silver_path):
    print("[INFO] Merging into existing Silver table")
    
    delta = DeltaTable.forPath(spark, silver_path)

    (delta.alias("t")
     .merge(df_upsert.alias("s"), "t.license_id = s.license_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    print("[INFO] Creating Silver table")
    df_upsert.write.format("delta").mode("overwrite").save(silver_path)

# 11. Validation

In [ ]:
df_check = spark.read.format("delta").load(silver_path)
df_check.show(5)

print("[SUCCESS] Silver license_keys pipeline complete")